In [0]:
-- Encontrar duplicados usando funciones ventana
-- Ventaja: muestra todas las filas duplicadas con sus detalles completos

WITH meetings_with_duplicates AS (
  SELECT 
    'silver_meetings_sessions' AS table_name,
    session_key,
    meeting_key,
    season,
    meeting_name,
    location,
    ROW_NUMBER() OVER (PARTITION BY session_key ORDER BY session_key) AS row_num,
    COUNT(*) OVER (PARTITION BY session_key) AS duplicate_count
  FROM workspace.formula_1.silver_dim_meetings_sessions
),
drivers_with_duplicates AS (
  SELECT 
    'silver_drivers' AS table_name,
    session_key,
    driver_number,
    driver_name,
    team_name,
    ROW_NUMBER() OVER (PARTITION BY session_key, driver_number ORDER BY session_key) AS row_num,
    COUNT(*) OVER (PARTITION BY session_key, driver_number) AS duplicate_count
  FROM workspace.formula_1.silver_dim_drivers
),
laps_with_duplicates AS (
  SELECT 
    'silver_laps' AS table_name,
    session_key,
    driver_number,
    lap_number,
    lap_duration_seconds,
    ROW_NUMBER() OVER (PARTITION BY session_key, driver_number, lap_number ORDER BY session_key) AS row_num,
    COUNT(*) OVER (PARTITION BY session_key, driver_number, lap_number) AS duplicate_count
  FROM workspace.formula_1.silver_fact_laps
)

-- Combinar resultados: solo filas donde duplicate_count > 1
SELECT 
  table_name,
  session_key,
  NULL AS driver_number,
  NULL AS lap_number,
  meeting_name AS detail_1,
  location AS detail_2,
  NULL AS detail_3,
  row_num,
  duplicate_count
FROM meetings_with_duplicates
WHERE duplicate_count > 1

UNION ALL

SELECT 
  table_name,
  session_key,
  driver_number,
  NULL AS lap_number,
  driver_name AS detail_1,
  team_name AS detail_2,
  NULL AS detail_3,
  row_num,
  duplicate_count
FROM drivers_with_duplicates
WHERE duplicate_count > 1

UNION ALL

SELECT 
  table_name,
  session_key,
  driver_number,
  lap_number,
  NULL AS detail_1,
  NULL AS detail_2,
  CAST(lap_duration_seconds AS STRING) AS detail_3,
  row_num,
  duplicate_count
FROM laps_with_duplicates
WHERE duplicate_count > 1

ORDER BY table_name, session_key, driver_number, lap_number, row_num;








-----------------TABLA workspace.formula_1.silver_dim_meetings_sessions-----------------
WITH parsed_meetings AS (
    SELECT
        TRY_CAST(get_json_object(raw_json, '$.meeting_key') AS INT) AS meeting_key,
        get_json_object(raw_json, '$.meeting_name') AS meeting_name,
        get_json_object(raw_json, '$.meeting_official_name') AS meeting_official_name,
        get_json_object(raw_json, '$.location') AS location,
        get_json_object(raw_json, '$.country_name') AS country_name,
        get_json_object(raw_json, '$.country_code') AS country_code,
        TRY_CAST(get_json_object(raw_json, '$.circuit_key') AS INT) AS circuit_key,
        get_json_object(raw_json, '$.circuit_short_name') AS circuit_short_name,
        get_json_object(raw_json, '$.circuit_type') AS circuit_type,
        TRY_CAST(get_json_object(raw_json, '$.is_cancelled') AS BOOLEAN) AS is_cancelled,
        TRY_CAST(get_json_object(raw_json, '$.year') AS INT) AS year
    FROM
        workspace.formula_1.bronze_meetings
    WHERE
        get_json_object(raw_json, "$.error") IS NULL
),
parsed_sessions AS (
    SELECT
        TRY_CAST(get_json_object(raw_json, '$.session_key') AS INT) AS session_key,
        TRY_CAST(get_json_object(raw_json, '$.meeting_key') AS INT) AS meeting_key,
        get_json_object(raw_json, '$.session_name') AS session_name,
        get_json_object(raw_json, '$.session_type') AS session_type,
        TO_TIMESTAMP(get_json_object(raw_json, '$.date_start')) AS session_start_time,
        TO_TIMESTAMP(get_json_object(raw_json, '$.date_end')) AS session_end_time,
        get_json_object(raw_json, '$.gmt_offset') AS gmt_offset
    FROM
        workspace.formula_1.bronze_sessions
    WHERE
        get_json_object(raw_json, "$.error") IS NULL
)
SELECT
    ses.session_key,
    met.meeting_key,
    met.year AS season,
    met.meeting_name,
    met.meeting_official_name,
    met.location,
    met.country_name,
    met.country_code,
    met.circuit_key,
    met.circuit_short_name,
    met.circuit_type,
    met.is_cancelled AS is_meeting_cancelled,
    ses.session_name,
    ses.session_type,
    ses.session_start_time,
    ses.session_end_time,
    ses.gmt_offset
FROM
    parsed_meetings met
INNER JOIN
    parsed_sessions ses ON met.meeting_key = ses.meeting_key
;

-----------------TABLA workspace.formula_1.silver_dim_drivers-----------------
WITH parsed_drivers AS (
    SELECT
        get_json_object(raw_json, '$.driver_number') AS driver_number,
        get_json_object(raw_json, '$.full_name') AS full_name,
        get_json_object(raw_json, '$.first_name') AS first_name,
        get_json_object(raw_json, '$.last_name') AS last_name,
        get_json_object(raw_json, '$.last_name') AS last_name,
        get_json_object(raw_json, '$.name_acronym') AS name_acronym,
        get_json_object(raw_json, '$.team_name') AS team_name,
        get_json_object(raw_json, '$.team_colour') AS team_colour,
        get_json_object(raw_json, '$.meeting_key') AS meeting_key,
        get_json_object(raw_json, '$.session_key') AS session_key
    FROM
        workspace.formula_1.bronze_drivers
)
SELECT
    DISTINCT
        session_key, -- El piloto mapeado a la sesión específica (por si cambia de equipo a mitad de año)
        driver_number,
        name_acronym,
        initcap(full_name) AS driver_name, -- Normaliza "Pierre GASLY" -> "Pierre Gasly"
        team_name,
        CASE 
            WHEN team_colour IS NOT NULL THEN concat('#', team_colour) 
            ELSE '#FFFFFF' 
        END AS team_hex_colour
FROM
    parsed_drivers;

-----------------TABLA workspace.formula_1.silver_fact_laps-----------------
WITH parsed_laps AS (
  SELECT 
    year,
    month,
    from_json(raw_json, 'lap_number INT, meeting_key INT, session_key INT, driver_number INT, lap_duration DOUBLE, duration_sector_1 DOUBLE, duration_sector_2 DOUBLE, duration_sector_3 DOUBLE, i1_speed INT, i2_speed INT, st_speed INT, is_pit_out_lap BOOLEAN, date_start STRING') AS l
  FROM workspace.formula_1.bronze_laps
)
SELECT 
  l.session_key,
  l.meeting_key,
  l.driver_number,
  l.lap_number,
  to_timestamp(l.date_start) AS lap_start_time,
  l.lap_duration AS lap_duration_seconds,
  l.duration_sector_1 AS s1_seconds,
  l.duration_sector_2 AS s2_seconds,
  l.duration_sector_3 AS s3_seconds,
  l.i1_speed AS speed_trap_1,
  l.i2_speed AS speed_trap_2,
  l.st_speed AS top_speed,
  l.is_pit_out_lap,
  -- Métrica de ingeniería de características rápida
  CASE 
    WHEN l.lap_duration = MIN(l.lap_duration) OVER(PARTITION BY l.session_key, l.driver_number) THEN true 
    ELSE false 
  END AS is_driver_personal_best,
  l.year,
  l.month
FROM parsed_laps l
WHERE l.lap_duration IS NOT NULL 
  AND l.lap_duration > 50.0 -- Filtro de calidad: eliminar vueltas corruptas o absurdamente cortas
  AND l.lap_duration < 300.0; -- Eliminar outliers (vueltas bajo bandera roja o autos detenidos)